# 07 — Hyper-parameter tuning with nested cross-validation

**Purpose.** Answer, honestly, one question: *does tuning Random Forest and XGBoost beat their default settings by more than noise?* — and, if so, produce the final hyper-parameters.

**Reads:** `../data/raw/dataset.xlsx`, `../data/processed/X_raw_train.csv`, `../results/fs_summary.csv` · **Writes:** `../results/tune_*.csv`, `tune_oof.npz`, `tuning_final_params.json`

**Fixed by earlier notebooks:** 14 raw features (notebook 06), no class weighting (notebook 05), the same 5 outer folds as notebooks 04–06.

**Why "nested" cross-validation.** If you search for the best settings using the same folds you report the score on, the reported score is optimistic: the winner of a search is partly the luckiest candidate. This is a common error: it makes tuned models look better than they are, and a "tuned" model can even score *worse* than the default under honest evaluation. Nested CV removes the problem:

* **Outer loop (5 folds, same as before):** the validation patients are never seen by the search.
* **Inner loop (3 folds inside each outer training part):** this is where candidate settings are compared and the best one is picked.
* The chosen setting is refit on the whole outer training part and scored once on the outer validation patients.

The outer scores therefore estimate the performance of *the whole tuning procedure*, which is what a paper should report.

**Design**

| Item | Choice |
|---|---|
| Search method | Random search; **trial 0 is always the default setting**, followed by 20 random settings (`N_TRIALS = 20`), so the search can never be worse than the default on the inner score |
| Inner score | Mean macro-F1 over 3 inner folds (identical inner folds for every candidate → paired) |
| Budget | 21 candidates × 3 inner fits = 63 fits per model per outer fold. **Notebook 09 (nature-inspired optimisation) must use the same budget**, otherwise the comparison with random search is meaningless |
| Models | Random Forest, XGBoost; no class weights; 14 raw features |

**Pre-specified decision rule.** For each model separately, tuned settings replace the defaults only if the 95% paired-bootstrap interval of the macro-F1 gain (tuned − default, pooled out-of-fold) **excludes 0** *and* tuned wins in **at least 4 of 5 outer folds**. Otherwise the defaults stay. Expectation stated in advance: with defaults already near 0.823 / 0.817 and a flat landscape, gains above 0.005 are unlikely; a "no significant gain" result is a legitimate, reportable finding.

**Honest caveats**
* **Mild inner-loop leakage.** The imputer and scaler of each outer fold were fit on the whole outer training part, including the inner validation rows. This can make inner scores very slightly optimistic and could bias *which* candidate wins, but it cannot affect the outer scores, which stay clean.
* A random search of 20 candidates in a 7–9-dimensional space samples it sparsely; a negative result means "no easy gain within this budget", not "tuning can never help".

## 0. Setup

**What this does:** imports libraries, the shared preprocessing/metric helpers, and prints versions.

In [1]:
import sys, os, time, json, warnings
sys.path.append('..')

import numpy as np
import pandas as pd
import sklearn, xgboost, scipy
from scipy.stats import loguniform, uniform
from sklearn.model_selection import StratifiedKFold, ParameterSampler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

from src.preprocessing.cv_pipeline import preprocess_fold, load_cv_pool
from src.evaluation.metrics import (macro_f1_fast, per_class_prf, brier_multiclass, ece_top_label,
                                    mcnemar_exact, holm, bootstrap_f1_matrix)

print("pandas", pd.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__,
      "| scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__)
os.makedirs('../results', exist_ok=True)

pandas 3.0.5 | numpy 2.5.2 | scipy 1.18.1 | scikit-learn 1.9.0 | xgboost 3.4.1


## 1. Load the CV pool and the 5 outer folds

**What this does:** loads the 9,668 training patients (never the test set), verifies they are identical to notebook 02's training set, and recreates the same 5 outer folds as notebooks 04–06.

In [2]:
pool = load_cv_pool("../data/raw/dataset.xlsx")
ref_idx = pd.read_csv('../data/processed/y_train.csv', index_col=0).index
assert pool.shape == (9668, 17) and pool.index.equals(ref_idx)
y_global = LabelEncoder().fit_transform(pool['Disease'])
classes = sorted(pool['Disease'].unique()); K = len(classes)
AS, REA = classes.index('Ankylosing Spondylitis'), classes.index('Reactive Arthritis')

N_SPLITS = 5
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42).split(pool, pool['Disease']))
cover = np.zeros(len(pool), int)
for tr, va in folds:
    cover[va] += 1
assert (cover == 1).all()
print("pool OK:", pool.shape, "| outer fold sizes:", [(len(tr), len(va)) for tr, va in folds])

pool OK: (9668, 17) | outer fold sizes: [(7734, 1934), (7734, 1934), (7734, 1934), (7735, 1933), (7735, 1933)]


## 2. Search spaces, defaults and budget

**What this does:** defines, for each model, the *default* setting (exactly what notebooks 04–06 used) and the ranges that random search draws from.

**Random Forest ranges**
* `n_estimators` 150–600 trees; `max_depth` unlimited or 8–24; `min_samples_leaf` 1–8 and `min_samples_split` 2–10 (larger = smoother trees); `max_features` (how many features each split may look at: √p, log₂p, or 30–70%); `max_samples` (fraction of patients each tree sees: all, 85% or 70%); `criterion` gini or entropy.

**XGBoost ranges**
* `n_estimators` 150–600; `learning_rate` 0.02–0.3 (log scale); `max_depth` 3–8; `min_child_weight` 1–10; `subsample` 0.6–1.0 and `colsample_bytree` 0.5–1.0 (row/column sampling per tree); `reg_lambda` 0.1–10 and `reg_alpha` 0.001–1 (log scales; L2/L1 regularisation); `gamma` 0–1 (minimum gain to split).

**Budget:** `N_TRIALS = 20` random candidates + the default per model per outer fold, each scored with 3 inner fits → 63 fits. Expect roughly 30–45 minutes in total (XGBoost is the slower of the two).

In [3]:
N_TRIALS = 20

models = {
    'Random Forest': dict(
        make=lambda **p: RandomForestClassifier(random_state=42, n_jobs=-1, **p),
        default={'n_estimators': 300},
        space={'n_estimators': [150, 250, 400, 600], 'max_depth': [None, 8, 12, 16, 24],
               'min_samples_leaf': [1, 2, 3, 5, 8], 'min_samples_split': [2, 5, 10],
               'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7], 'max_samples': [None, 0.85, 0.7],
               'criterion': ['gini', 'entropy']}),
    'XGBoost': dict(
        make=lambda **p: XGBClassifier(objective='multi:softprob', eval_metric='mlogloss', random_state=42, n_jobs=-1, **p),
        default={'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1},
        space={'n_estimators': [150, 250, 400, 600], 'learning_rate': loguniform(0.02, 0.3),
               'max_depth': [3, 4, 5, 6, 7, 8], 'min_child_weight': [1, 3, 5, 10],
               'subsample': uniform(0.6, 0.4), 'colsample_bytree': uniform(0.5, 0.5),
               'reg_lambda': loguniform(0.1, 10), 'reg_alpha': loguniform(1e-3, 1),
               'gamma': [0, 0.1, 0.5, 1]}),
}
def jsonable(p):
    return {k: (None if v is None else (int(v) if isinstance(v, (np.integer,)) else float(v) if isinstance(v, (np.floating,)) else v))
            for k, v in p.items()}
keys = [f"{m}|{v}" for m in models for v in ('default', 'tuned')]
print("candidates per model per outer fold:", N_TRIALS + 1, "| inner fits per model per fold:", (N_TRIALS + 1) * 3)

candidates per model per outer fold: 21 | inner fits per model per fold: 63


## 3. The nested cross-validation loop

**What this does, for each outer fold and each model:**
1. Preprocess on the outer training part only (`preprocess_fold`), keep the 14 raw features.
2. Build the candidate list: the default plus 20 random settings (a different random draw in each fold, reproducible through the seed).
3. Score every candidate by 3-fold inner CV on the outer training part (same inner folds for all candidates).
4. Pick the candidate with the best inner score (ties go to the default, which is first).
5. Refit **both** the default and the winner on the whole outer training part and predict the outer validation patients.

Every candidate's inner score is stored (needed later to compare with notebook 09's optimiser).

**Checks:** no missing values after preprocessing; encoded labels equal the global encoding; model classes are 0–6.

**What to watch:** a progress line per model per fold with its time.

In [4]:
oof_proba = {k: np.zeros((len(pool), K)) for k in keys}
y_all = np.full(len(pool), -1)
fold_rows, trial_rows, best_rows = [], [], []

for fold, (tr, va) in enumerate(folds, 1):
    ft, fv = pool.iloc[tr], pool.iloc[va]
    Xr_tr, Xr_va, _, _, y_tr, y_va = preprocess_fold(ft, fv)
    assert Xr_tr.isna().sum().sum() == 0 and Xr_va.isna().sum().sum() == 0 and Xr_tr.shape[1] == 14
    assert (y_va.to_numpy() == y_global[va]).all() and (y_tr.to_numpy() == y_global[tr]).all()
    Xt, Xv, ytr, yva = Xr_tr, Xr_va, y_tr.to_numpy(), y_va.to_numpy()
    y_all[va] = yva
    inner = list(StratifiedKFold(3, shuffle=True, random_state=1).split(Xt, ytr))
    for mname, cfg in models.items():
        t0 = time.time()
        trials = [cfg['default']] + list(ParameterSampler(cfg['space'], n_iter=N_TRIALS, random_state=42 + fold))
        scores = []
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            for t_i, p in enumerate(trials):
                sc = [macro_f1_fast(ytr[iva], cfg['make'](**p).fit(Xt.iloc[itr], ytr[itr]).predict(Xt.iloc[iva]).astype(int))
                      for itr, iva in inner]
                scores.append(float(np.mean(sc)))
                trial_rows.append({'fold': fold, 'model': mname, 'trial': t_i, 'inner_macro_f1': scores[-1],
                                   'params': json.dumps(jsonable(p))})
            best_i = int(np.argmax(scores))
            outer = {}
            for variant, p in (('default', cfg['default']), ('tuned', trials[best_i])):
                m = cfg['make'](**p).fit(Xt, ytr)
                assert list(m.classes_) == list(range(K))
                P = m.predict_proba(Xv); pred = P.argmax(1)
                oof_proba[f"{mname}|{variant}"][va] = P
                pr, rc, f1 = per_class_prf(yva, pred, K)
                outer[variant] = f1_score(yva, pred, average='macro')
                fold_rows.append({'fold': fold, 'model': mname, 'variant': variant, 'key': f"{mname}|{variant}",
                                  'macro_f1': outer[variant], 'balanced_accuracy': rc.mean(),
                                  'AS_f1': f1[AS], 'ReA_f1': f1[REA]})
        best_rows.append({'fold': fold, 'model': mname, 'best_trial': best_i, 'default_was_best': best_i == 0,
                          'inner_default': scores[0], 'inner_best': scores[best_i], 'inner_median': float(np.median(scores)),
                          'default_inner_rank': int((np.array(scores) > scores[0]).sum()) + 1,
                          'outer_default': outer['default'], 'outer_tuned': outer['tuned'],
                          'best_params': json.dumps(jsonable(trials[best_i]))})
        print(f"fold {fold}/{N_SPLITS} {mname}: {time.time() - t0:.0f}s  (inner default {scores[0]:.4f} -> best {scores[best_i]:.4f}; outer default {outer['default']:.4f} -> tuned {outer['tuned']:.4f})")

fold_df = pd.DataFrame(fold_rows); trials_df = pd.DataFrame(trial_rows); best_df = pd.DataFrame(best_rows)
oof_pred = {k: v.argmax(1) for k, v in oof_proba.items()}
print("done.")

fold 1/5 Random Forest: 65s  (inner default 0.8143 -> best 0.8217; outer default 0.8154 -> tuned 0.8226)
fold 1/5 XGBoost: 117s  (inner default 0.8164 -> best 0.8256; outer default 0.8089 -> tuned 0.8173)
fold 2/5 Random Forest: 84s  (inner default 0.8174 -> best 0.8245; outer default 0.8198 -> tuned 0.8231)
fold 2/5 XGBoost: 112s  (inner default 0.8152 -> best 0.8254; outer default 0.8234 -> tuned 0.8253)
fold 3/5 Random Forest: 84s  (inner default 0.8180 -> best 0.8201; outer default 0.8316 -> tuned 0.8301)
fold 3/5 XGBoost: 142s  (inner default 0.8116 -> best 0.8235; outer default 0.8220 -> tuned 0.8361)
fold 4/5 Random Forest: 1171s  (inner default 0.8206 -> best 0.8223; outer default 0.8192 -> tuned 0.8175)
fold 4/5 XGBoost: 133s  (inner default 0.8157 -> best 0.8258; outer default 0.8170 -> tuned 0.8187)
fold 5/5 Random Forest: 91s  (inner default 0.8188 -> best 0.8225; outer default 0.8292 -> tuned 0.8298)
fold 5/5 XGBoost: 228s  (inner default 0.8197 -> best 0.8261; outer defau

## 4. Integrity checks and reproduction of the earlier default results

**What this does:**
* every patient has a label and every probability row sums to 1;
* the **default** variant must reproduce the earlier `raw` results of notebook 06 exactly (same folds, features, model settings). This proves the nested machinery did not disturb the baseline.

**If any difference exceeds 0.0005, stop and investigate.**

In [5]:
assert (y_all == y_global).all()
for k, P in oof_proba.items():
    assert np.allclose(P.sum(1), 1.0, atol=1e-6), k

ref = pd.read_csv('../results/fs_summary.csv', index_col=0)['macro_f1_mean']
gmean = fold_df.groupby('key').macro_f1.mean()
rep = pd.DataFrame([{'model': m, 'notebook_06_raw': ref[f"{m}|raw"], 'this_notebook_default': gmean[f"{m}|default"]} for m in models])
rep['diff'] = rep['this_notebook_default'] - rep['notebook_06_raw']
print(rep.round(5).to_string(index=False))
print("\nmax |diff| =", rep['diff'].abs().max().round(6))
if rep['diff'].abs().max() > 0.0005:
    print("WARNING: default does not reproduce notebook 06 - investigate.")

        model  notebook_06_raw  this_notebook_default  diff
Random Forest          0.82303                0.82303   0.0
      XGBoost          0.81741                0.81741   0.0

max |diff| = 0.0


## 5. How much of the tuning gain was real? (inner vs outer)

**What this does:** for each model and outer fold shows
* `inner_default`, `inner_best`: the inner-CV score of the default and of the winning candidate — what the search *believed*;
* `outer_default`, `outer_tuned`: what actually happened on the unseen outer validation patients;
* `inner_gain` = inner_best − inner_default (the search's claimed improvement) and `outer_gain` = outer_tuned − outer_default (the real one);
* `default_inner_rank`: the default's rank among the 21 candidates by inner score (1 = best). A rank in the middle or top third means the landscape is flat: the default is already about as good as a typical random setting.

**How to read it:** if `inner_gain` is consistently positive but `outer_gain` hovers around 0 (or is negative), the search is chasing noise — the classic selection optimism that nested CV exists to expose.

In [6]:
view = best_df.copy()
view['inner_gain'] = view['inner_best'] - view['inner_default']
view['outer_gain'] = view['outer_tuned'] - view['outer_default']
print(view[['fold', 'model', 'default_was_best', 'default_inner_rank', 'inner_default', 'inner_best', 'inner_gain',
            'outer_default', 'outer_tuned', 'outer_gain']].round(4).to_string(index=False))
print()
print(view.groupby('model')[['inner_gain', 'outer_gain']].mean().round(4))
print("\nmean selection optimism (inner_gain - outer_gain):")
print((view.groupby('model')['inner_gain'].mean() - view.groupby('model')['outer_gain'].mean()).round(4))

 fold         model  default_was_best  default_inner_rank  inner_default  inner_best  inner_gain  outer_default  outer_tuned  outer_gain
    1 Random Forest             False                  14         0.8143      0.8217      0.0074         0.8154       0.8226      0.0073
    1       XGBoost             False                  16         0.8164      0.8256      0.0092         0.8089       0.8173      0.0084
    2 Random Forest             False                  10         0.8174      0.8245      0.0071         0.8198       0.8231      0.0034
    2       XGBoost             False                  15         0.8152      0.8254      0.0101         0.8234       0.8253      0.0019
    3 Random Forest             False                   8         0.8180      0.8201      0.0021         0.8316       0.8301     -0.0015
    3       XGBoost             False                  19         0.8116      0.8235      0.0119         0.8220       0.8361      0.0141
    4 Random Forest             False    

## 6. Did the folds agree on the best settings?

**What this does:** prints the winning parameters of each outer fold. If different folds pick very different settings, no single setting is clearly best — more evidence that the landscape is flat and any tuned model is one of many near-equivalent choices. If most folds keep the default (`trial 0`), the search found nothing better.

In [7]:
for m in models:
    print(f"\n=== {m} ===")
    for _, r in best_df[best_df.model == m].iterrows():
        tag = ' (default)' if r['default_was_best'] else ''
        print(f"fold {r['fold']}: trial {r['best_trial']:>2}{tag}: {r['best_params']}")


=== Random Forest ===
fold 1: trial  1: {"n_estimators": 150, "min_samples_split": 10, "min_samples_leaf": 3, "max_samples": 0.7, "max_features": 0.5, "max_depth": 16, "criterion": "gini"}
fold 2: trial  3: {"n_estimators": 600, "min_samples_split": 10, "min_samples_leaf": 1, "max_samples": null, "max_features": 0.3, "max_depth": 24, "criterion": "gini"}
fold 3: trial  3: {"n_estimators": 150, "min_samples_split": 2, "min_samples_leaf": 2, "max_samples": 0.85, "max_features": 0.5, "max_depth": 8, "criterion": "entropy"}
fold 4: trial  3: {"n_estimators": 400, "min_samples_split": 5, "min_samples_leaf": 3, "max_samples": 0.7, "max_features": 0.5, "max_depth": 12, "criterion": "gini"}
fold 5: trial 11: {"n_estimators": 400, "min_samples_split": 10, "min_samples_leaf": 2, "max_samples": null, "max_features": 0.5, "max_depth": null, "criterion": "gini"}

=== XGBoost ===
fold 1: trial  1: {"colsample_bytree": 0.5575272831948894, "gamma": 1, "learning_rate": 0.026428173189806673, "max_depth

## 7. Results and the pre-specified decision

**What this does:** one row per (model, variant) with the 5-fold mean ± std, the pooled out-of-fold macro-F1 with a 95% bootstrap interval, and the other metrics; then compares tuned against default **on the same patients**:
* `diff` and `ci_low`/`ci_high`: pooled macro-F1 gain (tuned − default) and its 95% paired-bootstrap interval;
* `folds_tuned_better`: number of outer folds (of 5) where tuned beats default;
* McNemar exact test on accuracy (`b` = only tuned right, `c` = only default right), Holm-corrected over the 2 models;
* `adopt`: the rule — interval excludes 0 on the positive side **and** ≥ 4 folds better.

If a model's tuned variant equals the default in a fold (default won the inner search), that fold contributes a zero difference, which is correct: the procedure chose to keep the default.

In [8]:
keys_b, F = bootstrap_f1_matrix(y_all, oof_pred, B=2000, seed=42)
kpos = {k: j for j, k in enumerate(keys_b)}
gk = fold_df.groupby('key')
rows_s = []
for k in keys:
    m, v = k.split('|')
    pr, rc, f1 = per_class_prf(y_all, oof_pred[k], K)
    rows_s.append({'model': m, 'variant': v, 'macro_f1_mean': gk.macro_f1.mean()[k], 'macro_f1_std': gk.macro_f1.std()[k],
                   'oof_macro_f1': macro_f1_fast(y_all, oof_pred[k]),
                   'ci_low': np.percentile(F[:, kpos[k]], 2.5), 'ci_high': np.percentile(F[:, kpos[k]], 97.5),
                   'AS_f1': f1[AS], 'ReA_f1': f1[REA], 'balanced_acc': rc.mean(),
                   'brier': brier_multiclass(y_all, oof_proba[k]), 'ece': ece_top_label(y_all, oof_proba[k])})
summary = pd.DataFrame(rows_s)
print(summary.round(4).to_string(index=False))

piv = fold_df.pivot(index='fold', columns='key', values='macro_f1')
cmp_rows = []
for m in models:
    A, B = f"{m}|tuned", f"{m}|default"
    d = F[:, kpos[A]] - F[:, kpos[B]]
    b, c, p = mcnemar_exact(y_all, oof_pred[A], oof_pred[B])
    fd = piv[A] - piv[B]
    cmp_rows.append({'model': m, 'diff': macro_f1_fast(y_all, oof_pred[A]) - macro_f1_fast(y_all, oof_pred[B]),
                     'ci_low': np.percentile(d, 2.5), 'ci_high': np.percentile(d, 97.5),
                     'folds_tuned_better': int((fd > 1e-12).sum()), 'folds_identical': int((fd.abs() <= 1e-12).sum()),
                     'mcnemar_b': b, 'mcnemar_c': c, 'p_raw': p})
cmp_df = pd.DataFrame(cmp_rows)
cmp_df['p_holm'] = holm(cmp_df['p_raw'])
cmp_df['adopt'] = (cmp_df['ci_low'] > 0) & (cmp_df['folds_tuned_better'] >= 4)
print("\nTuned vs default (pre-registered rule):")
print(cmp_df.drop(columns='p_raw').round(4).to_string(index=False))
adopt = dict(zip(cmp_df['model'], cmp_df['adopt']))
print("\nDECISION:", {m: ('use tuned settings' if a else 'keep defaults') for m, a in adopt.items()})

        model variant  macro_f1_mean  macro_f1_std  oof_macro_f1  ci_low  ci_high  AS_f1  ReA_f1  balanced_acc  brier    ece
Random Forest default         0.8230        0.0070        0.8230  0.8139   0.8316 0.6623  0.6563        0.8114 0.2196 0.0370
Random Forest   tuned         0.8246        0.0053        0.8247  0.8161   0.8333 0.6578  0.6676        0.8148 0.2124 0.0176
      XGBoost default         0.8174        0.0057        0.8175  0.8090   0.8258 0.6673  0.6439        0.8108 0.2308 0.0575
      XGBoost   tuned         0.8254        0.0078        0.8254  0.8168   0.8336 0.6671  0.6703        0.8159 0.2098 0.0153

Tuned vs default (pre-registered rule):
        model   diff  ci_low  ci_high  folds_tuned_better  folds_identical  mcnemar_b  mcnemar_c  p_holm  adopt
Random Forest 0.0017 -0.0021   0.0055                   3                0        153        142  0.5605  False
      XGBoost 0.0079  0.0025   0.0132                   5                0        333        249  0.0011   Tru

## 8. Final hyper-parameters for the test-set evaluation

**What this does:** produces the settings notebook 10 will use for the single, final evaluation.
* For a model where tuning **passed** the rule: run the same random search once more on the **whole** training pool (9,668 patients, using the preprocessed training file from notebook 02) with a **5-fold** inner CV, and record the winner. The nested loop above estimates how good this *procedure* is; this step applies it once to all training data.
* For a model where tuning **did not pass**: record the defaults. Nothing is searched.

Output: `../results/tuning_final_params.json`. If no model passed, this cell is quick and nothing is tuned.

In [9]:
final = {}
Xfull = pd.read_csv('../data/processed/X_raw_train.csv', index_col=0)
yfull = pd.read_csv('../data/processed/y_train.csv', index_col=0).iloc[:, 0].to_numpy()
assert Xfull.shape == (9668, 14) and Xfull.index.equals(pool.index) and (yfull == y_global).all()
inner5 = list(StratifiedKFold(5, shuffle=True, random_state=1).split(Xfull, yfull))
for mname, cfg in models.items():
    if not adopt[mname]:
        final[mname] = {'source': 'defaults retained (tuning not shown to help under nested CV)', 'params': jsonable(cfg['default'])}
        print(f"{mname}: defaults retained -> {jsonable(cfg['default'])}")
        continue
    t0 = time.time()
    trials = [cfg['default']] + list(ParameterSampler(cfg['space'], n_iter=N_TRIALS, random_state=999))
    scs = []
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for p in trials:
            scs.append(float(np.mean([macro_f1_fast(yfull[iva], cfg['make'](**p).fit(Xfull.iloc[itr], yfull[itr]).predict(Xfull.iloc[iva]).astype(int))
                                      for itr, iva in inner5])))
    bi = int(np.argmax(scs))
    final[mname] = {'source': f'random search on the full training pool, 5-fold inner CV, {N_TRIALS} candidates + default',
                    'params': jsonable(trials[bi]), 'inner_cv_macro_f1': scs[bi], 'inner_cv_macro_f1_default': scs[0]}
    pd.DataFrame({'trial': range(len(trials)), 'inner_macro_f1': scs, 'params': [json.dumps(jsonable(p)) for p in trials]}) \
        .to_csv(f"../results/tune_final_trials_{mname.replace(' ', '_')}.csv", index=False)
    print(f"{mname}: tuned in {time.time() - t0:.0f}s -> {jsonable(trials[bi])} (inner CV {scs[bi]:.4f} vs default {scs[0]:.4f})")
with open('../results/tuning_final_params.json', 'w') as f:
    json.dump(final, f, indent=2)
print("\nSaved ../results/tuning_final_params.json")

Random Forest: defaults retained -> {'n_estimators': 300}
XGBoost: tuned in 2395s -> {'colsample_bytree': 0.5702109964222462, 'gamma': 1, 'learning_rate': 0.02020431380302154, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 600, 'reg_alpha': 0.0013859108660520538, 'reg_lambda': 0.9559973589036749, 'subsample': 0.8058757277304458} (inner CV 0.8253 vs default 0.8166)

Saved ../results/tuning_final_params.json


## 9. Save the results

**What this does:** writes all tables, every inner-loop candidate score (reused by notebook 09 for a like-for-like comparison) and the out-of-fold probabilities to `../results/`.

In [10]:
fold_df.drop(columns=['key']).to_csv('../results/tune_fold_scores.csv', index=False)
summary.to_csv('../results/tune_summary.csv', index=False)
cmp_df.drop(columns=['p_raw']).to_csv('../results/tune_tuned_vs_default.csv', index=False)
best_df.to_csv('../results/tune_best_per_fold.csv', index=False)
trials_df.to_csv('../results/tune_inner_trials.csv', index=False)
np.savez_compressed('../results/tune_oof.npz', y=y_all, classes=np.array(classes),
                    **{k.replace('|', '__').replace(' ', '_'): v.astype(np.float32) for k, v in oof_proba.items()})
print(sorted(f for f in os.listdir('../results') if f.startswith('tune')))

['tune_best_per_fold.csv', 'tune_final_trials_XGBoost.csv', 'tune_fold_scores.csv', 'tune_inner_trials.csv', 'tune_oof.npz', 'tune_summary.csv', 'tune_tuned_vs_default.csv']


## Summary — what this notebook establishes

* An honest estimate of what tuning buys for Random Forest and XGBoost on this data (nested CV), with the selection optimism made visible (section 5).
* Whether tuned settings pass the pre-specified rule (section 7), and the final hyper-parameters for notebook 10 (section 8).
* A saved record of every candidate's inner score, so notebook 09 can be compared against random search under exactly the same budget and folds.